# Parameter-recovery experiment data

Load all available FNN parameter-search JSONL results into one tidy DataFrame. Each row is one training epoch; `parameter_trace` contains the learned-versus-true physical parameters recorded at that epoch. This notebook intentionally starts with loading and completeness checks only.

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd() if (Path.cwd() / 'derived').is_dir() else Path.cwd().parent
RESULTS_DIR = REPO_ROOT / 'derived' / 'results' / 'parameter_recovery'

FILENAME_PATTERN = re.compile(
    r'^paramrecovery_(?P<dynamic>diffusion|wave|coupled_oscillator)_'
    r'(?P<topology>[^_]+)_'
    r'(?P<parameter>gamma|omega|force_scale|topology)init(?P<initial_value>[^_]+)_'
    r'models(?P<models>\d+)_epochs(?P<epochs>\d+)_nodes(?P<num_nodes>\d+)_'
    r'episodes(?P<num_episodes>\d+)_bins(?P<num_bins>\d+)_events(?P<events_per_bin>\d+)_'
    r'dropint(?P<raindrop_interval>\d+)_tau(?P<event_threshold>[^_]+)_'
    r'trainroll(?P<rollout_train_steps>\d+)_rollhorizon(?P<rollout_horizon>\d+)_seed(?P<seed>\d+)\.jsonl$'
)


In [ ]:
def decode_numeric_tag(value: str) -> float:
    return float(value.replace('p', '.').replace('m', '-'))


def load_parameter_results(results_dir: Path) -> pd.DataFrame:
    paths = sorted(results_dir.glob('paramrecovery_*.jsonl'))
    if not paths:
        raise FileNotFoundError(f'No parameter-recovery JSONLs found in {results_dir}')

    records: list[dict] = []
    for path in paths:
        match = FILENAME_PATTERN.fullmatch(path.name)
        if match is None:
            raise ValueError(f'Unexpected parameter-recovery filename: {path.name}')

        file_meta = match.groupdict()
        for key in ('models', 'epochs', 'num_nodes', 'num_episodes', 'num_bins',
                    'events_per_bin', 'raindrop_interval', 'rollout_train_steps',
                    'rollout_horizon', 'seed'):
            file_meta[key] = int(file_meta[key])
        file_meta['initial_value'] = decode_numeric_tag(file_meta['initial_value'])
        file_meta['event_threshold'] = decode_numeric_tag(file_meta['event_threshold'])
        file_meta['source_file'] = path.name

        for line_number, line in enumerate(path.read_text().splitlines(), start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f'Invalid JSON in {path}:{line_number}') from exc
            record.update(file_meta)
            records.append(record)

    return pd.json_normalize(records, sep='.')


parameter_df = load_parameter_results(RESULTS_DIR)
parameter_df.head()


In [ ]:
# Availability and run completeness. A run has its final evaluation row once epoch == epochs.
print('Rows:', len(parameter_df))
print('Columns:', len(parameter_df.columns))

run_status = (
    parameter_df.groupby(['dynamic', 'topology', 'parameter', 'initial_value', 'seed', 'source_file'], observed=True)
    .agg(last_epoch=('epoch', 'max'), planned_epochs=('epochs', 'first'), rows=('epoch', 'size'))
    .reset_index()
)
run_status['complete'] = run_status['last_epoch'].eq(run_status['planned_epochs'])
display(run_status.sort_values(['dynamic', 'parameter', 'initial_value', 'seed']))

trace_columns = [column for column in parameter_df.columns if column.startswith('parameter_trace.')]
print('Parameter-trace columns:', trace_columns)
